# ⬆️ Towards realistic DFT + DMFT

Now that we have plenty of experience writing DMFT scripts, we will develop a DMFT script suitable for a realistic DFT+DMFT script, so that you can study realistic quantum embedding in combination with electronic structure bases on density-functional theory. In this tutorial, we will learn how to:

- use the ``Embedding`` class for defining the embedding descrption,
- introduce the **double-counting correction** to maintain the relative energy splitting between the correlated and non-correlated orbitals

By the end of this tutorial, you will learn how to write a DFT+DMFT script that can be used for any DFT+DMFT calculation for any correlated material. You will have the choice to study either:



- [**La$_{2}$CuO$_{4}$**](05s-la2cuo4.ipynb) is the parent compound of the high-temperature cuprate supercondcutors and exhibits a Mott insulating ground state driven by strong electron-electron interactions. In large energy window DFT+DMFT calculations, both copper and oxygen states are explicitly retained, allowing us to capture the full multi-orbital character of the electronic structure -- unlike single-band models (which we've studied so far) that downfold to a low-energy subspace. This approach reveals the formation of Hubbard bands and a charge-transfer gap, emphasizing the essential role of strong correlations and ligand hybridization in driving the insulating state.


- [**SrVO$_{3}$**](05s-srvo3.ipynb) is a correlated metail with a nominal $d^{1}$ configuration, often studied as a benchmark for moderately correlated systems. DFT+DMFT reveals the coexistence of coherent quasiparticle states at the Fermi level and incoherent Hubbard bands, highlighting the importance of dynamical correlations in systems that appear metallic within conventional band theory.

## 🧩 Quantum Embedding and Connecting to Electronic Structure Codes

We saw previously in the lectures that quantum embedding boils down to upfolding the self-energy that we compute from a quantum impurity problem back to the global electronic structure of a crystal. This means the lattice self-energy is obtained from

$$ \Sigma_{\mathrm{latt}}(\mathbf{k},\omega) = P^{\dagger}_{m\nu}(\mathbf{k})[\Sigma_{\mathrm{embed}}(\omega)]_{m m'} P_{m'\nu'}(\mathbf{k}), $$
where $m$ are orbital indices and $\nu$ are Kohn-Sham band indices. We refere to $P_{m\nu}(\mathbf{k})$ as a projector which projects the Kohn-Sham states onto a set of localized atomic orbitals. In practice, in order to connect to electronic structure codes like density-functional theory or many-body perturbation theory, we need the Kohn-Shame band structure ($\varepsilon_{\nu}(\mathbf{k})$ and the projector $P_{m\nu}(\mathbf{k})$. The TRIQS ecosystem supports interfaces (converters) to many different electronic structure codes available to the community. The currently supported DFT codes that have interfaces that suppport charge self-consistent DFT + DMFT are list below in the table. The projection scheme correponds to the program that is used to create the projectors mentioned above.

| DFT Code |  Basis | Projection Scheme(s) |
|:----------|:--------|:--------------------:|
| [Wien2k](http://susi.theochem.tuwien.ac.at) |  All-electron, full potential (LAPW+lo) | [Projectors (dmftproj)](https://triqs.github.io/dft_tools/latest/guide/conv_wien2k.html))  |
| [VASP](https://www.vasp.at)    | Pseudopotentials (PAW) | [Projectors (LOCPROJ](https://triqs.github.io/dft_tools/latest/guide/conv_vasp.html)), [Wannier90](https://triqs.github.io/dft_tools/latest/guide/conv_W90.html) |
| [Quantum Espresso](https://www.quantum-espresso.org) | Pseudopotentials | [Wannier90](https://triqs.github.io/dft_tools/latest/guide/conv_W90.html) |
| [Elk](https://elk.sourceforge.io) | All-electron, full potential (LAPW) | [Projectors](https://triqs.github.io/dft_tools/latest/guide/conv_elk.html) |

The tutorials for SrVO$_{3}$ and La$_{2}$CuO$_{4}$ where produced using the [Wien2k DFT code](), which is an all-electron, full potential code and often referred to as the gold standard for total energy calculations. TRIQS provides a Fortran90 program ``dmftproj`` which creates the projectors from the Wien2k data. We can then use the ``Wien2kConverter`` from our suite of DFT converters to prepare an HDF5 which contains all of the data that we need to run a DMFT calculation. In the following, we describe the input files for ``dmftproj`` for SrVO$_{3}$ and La$_{2}CuO$_{4}$. Additionally, we provide a utility program called ``init_dmftpr`` which helps the user prepare the input file for your material.

### 🧱 La$_{2}$CuO$_{4}$

#### DFT
In Wien2k, we can set up all the input files from only the Wien2k structure file which we provide in refernece data. To follow along, one can excute the following commands:

**Initialization**
```bash
init_lapw -prec 1 -vxc LDA
```

**Self-consistent field**
```bash
run_lapw -ec 0.00001 -cc 0.0001
```

For more details, we refer to the Wien2k documentation.

#### Projectors

To create the projectors, we need to run the following commands
```bash
x lapw1       # compute the KS eignvalues
x lapw2 -almd # compute coefficents in the local orbital basis
```

The ``dmftproj`` program takes the following input file which can be generated using ``init_dmftpr``:

```
!case.indmft
4         ! number of atoms in the structure file
2 1 2 2   ! multiplicity of each atom
3         ! the maximum angular quantum number l; we repeat the following block for each atom
cubic     ! (La) choice of spherical harmonics (cubic/spherical/custom)
0 0 1 0   ! Create a projector for s/p/d/f? 0 = No, 1 = Yes (do not treate as correlated), 2 = Yes (treat as correlated)
0 0 0 0   ! Split into irreps 0 = No, X = number of irreps
cubic     ! (Cu) cubic harmonics
0 0 2 0   ! Treat d-orbitals as correlated
0 0 0 0   ! No splitting
0         ! Do we have spin-orbit? 0 = No, 1 = Yes
cubic     ! (O) cubic harmonics
0 1 0 0   ! Create a projector for p
0 0 0 0   ! No splitting
cubic     ! (O) cubic harmonics
0 1 0 0   ! Create a projector for p
0 0 0 0   ! No splitting
-0.73 0.73 ! energy window around the Fermi level (in Ry)
 ```

### 🧱 SrVO3

In Wien2k, we can set up all the input files from only the Wien2k structure file which we provide in refernece data. To follow along, one can excute the following commands:

**Initialization**
```bash
init_lapw -prec 1 -vxc LDA
```

**Self-consistent field**
```bash
run_lapw -ec 0.00001 -cc 0.0001
```

For more details, we refer to the Wien2k documentation.

#### Projectors

To create the projectors, we need to run the following commands
```bash
x lapw1       # compute the KS eignvalues
x lapw2 -almd # compute coefficents in the local orbital basis
```

The ``dmftproj`` program takes the following input file which can be generated using ``init_dmftpr``:

```
3          ! number of atoms in the structure file
1 1 3      ! multiplicity of each atom
3          ! the maximum angular quantum number l; we repeat the following block for each atom
cubic      ! (V) choice of spherical harmonics (cubic/spherical/custom)
0 0 2 0    ! Treat d-orbitals as correlated
0 0 2 0    ! Split the d-orbitals into eg and t2g irreps 
01         ! choose only the t2g orbitals
0          ! No spin-orbit coupling
cubic      ! (Sr) cubic harmonics
1 0 0 0    ! Create a projector for s
0 0 0 0    ! No splitting
cubic      ! (O) cubic harmonics
0 1 0 0    ! Create a projector for p
0 0 0 0    ! No splitting
-0.73 0.73 ! energy window around the Fermi level (in Ry)
 ```

### ↔️ DFT Converters

While we are demonstrating the workflow for Wien2k + TRIQS, the general workflow is the same for all of the DFT codes that TRIQS has an interface to. After running the DFT code and creating the projectors either through the projection code or Wannier90, we must run a DFT converter. TRIQS provides:

- [Wannier90Converter]()
- [Wien2kConverter]()
- [VaspConverter]()

An example of how to run one these converters is below.

```python
from triqs_modest.converters import Wien2kConverter
Wien2kConveter("case").convert_dft_input()
```

This will produce a ``case.h5`` file which contains the group ``dft_input``.